# **Face Generation with PCA & Gaussian Mixture**  
Using `sklearn.datasets.load_digits()` as a proxy for faces, reduce dimensionality with **PCA**, then generate new samples using a **Gaussian Mixture Model**.
<hr>

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

In [2]:
digits = load_digits()
X, y = digits.data, digits.target
print ('Digits dataset: %d samples, each 8x8 pixels'%X.shape[0])
print ('Flattened feature dimension: %d'%X.shape[1])
print ('Classes: %d'%len(np.unique(y)))

Digits dataset: 1797 samples, each 8x8 pixels
Flattened feature dimension: 64
Classes: 10


In [3]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title('Label: %d'%y[i])
    ax.axis('off')
plt.suptitle('**Sample Digits**')
plt.tight_layout()
plt.show()

<Figure size 600x200 with 1 Axes>

In [4]:
print ('X shape: %s'%str(X.shape))

X shape: (1797, 64)


## **PCA Dimensionality Reduction**
<hr>

In [5]:
pca = PCA().fit(X)
print ('Explained variance ratio by component:')
for i in range(5):
    print ('PC%d:  %.3f'%(i+1, pca.explained_variance_ratio_[i]))

Explained variance ratio by component:
PC1:  0.149
PC2:  0.136
PC3:  0.118
PC4:  0.085
PC5:  0.059


In [6]:
plt.figure(figsize=(8, 5))
cumsum = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cumsum)+1), cumsum, 'bo-', markersize=3)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
plt.axvline(x=np.argmax(cumsum >= 0.95)+1, color='g', linestyle='--', label='Optimal dim')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('**PCA Explained Variance**')
plt.legend()
plt.grid(True)
plt.show()

<Figure size 700x400 with 1 Axes>

In [7]:
n_components = np.argmax(cumsum >= 0.95) + 1
print ('Reducing to %d components (95%% variance retained).'%n_components)
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X)
print ('Reduced shape: %s'%str(X_pca.shape))

Reducing to 30 components (95% variance retained).


## **Gaussian Mixture Model for Generation**
<hr>

In [8]:
print ('Fitting GaussianMixture with 50 components...')
gmm = GaussianMixture(n_components=50, covariance_type='full', random_state=42)
gmm.fit(X_pca)

Fitting GaussianMixture with 50 components...


In [9]:
print ('GMM converged: %s'%gmm.converged_)
print ('Log-likelihood: %.3f\n'%gmm.score(X_pca))

GMM converged: True
Log-likelihood: -4.682


## **Generate New Faces**
<hr>

In [10]:
print ('Generating 10 new samples...')
X_gen_pca = gmm.sample(10)[0]
print ('Generated sample shape: %s'%str(X_gen_pca.shape))

Generating 10 new samples...
Generated sample shape: (10, 30)


In [11]:
print ('Inverse transforming back to original space...')
X_gen = pca.inverse_transform(X_gen_pca)
print ('Reconstructed shape: %s\n'%str(X_gen.shape))

Inverse transforming back to original space...


In [12]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i in range(10):
    axes[0, i].imshow(X[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Original %d'%y[i])
    axes[1, i].imshow(X_gen[i].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Generated')
plt.suptitle('**Original vs Generated Digits**', fontsize=14)
plt.tight_layout()
plt.show()

<Figure size 800x400 with 2 Axes>

In [13]:
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_gen[i].reshape(8, 8), cmap='gray')
    ax.axis('off')
    ax.set_title('Generated %d'%(i+1))
plt.suptitle('**10 Newly Generated Digit Images**', fontsize=14)
plt.tight_layout()
plt.show()

<Figure size 800x400 with 1 Axes>

In [14]:
orig_mean = np.mean(X)
gen_mean = np.mean(X_gen)
print ('Average pixel value (original):  %.3f'%orig_mean)
print ('Average pixel value (generated): %.3f'%gen_mean)
if abs(orig_mean - gen_mean) < 0.02:
    print ('Close match ** generation works!')

Average pixel value (original):  0.133
Average pixel value (generated): 0.128
Close match — generation works!


<hr>
## **Summary**
- Used **PCA** to reduce 64-dimensional digit images to **30 components** (95% variance).
- Fitted a **Gaussian Mixture Model** to learn the latent distribution.
- Generated **10 new realistic-looking digit images** via sampling from the GMM.
- Average pixel intensity matches the original dataset, validating the generative approach.
- This technique scales to real face datasets with higher resolution.
<hr>